In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
# original['TotalCharges'].fillna(original['MonthlyCharges'] * original['tenure'], inplace=True)

In [5]:
original.drop('customerID', axis=1, inplace=True)
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [6]:
cats = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

nums = ['tenure', 'MonthlyCharges', 'TotalCharges']

target = 'Churn'

new_nums = []
nums_as_cat = []

In [7]:
for col in ["Contract", "InternetService", "PaymentMethod"]:

    mean_map = train.groupby(col)["MonthlyCharges"].mean()
    train[f"{col}ChargeMean"] = train[col].map(mean_map)
    test[f"{col}ChargeMean"] = test[col].map(mean_map)

    mean_map = train.groupby(col)["tenure"].mean()
    train[f"{col}TenureMean"] = train[col].map(mean_map)
    test[f"{col}TenureMean"] = test[col].map(mean_map)

In [8]:
for col in nums:
    freq = pd.concat([train[col], original[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, original]:
        df[f'Freq{col}'] = df[col].map(freq).fillna(0).astype('float32')
    new_nums.append(f'Freq{col}')

In [9]:
for col in cats:

    churn_mean = original.groupby(col)[target].mean()
    churn_std = original.groupby(col)[target].std()

    train[f"{col}ChurnMean"] = train[col].map(churn_mean)
    test[f"{col}ChurnMean"] = test[col].map(churn_mean)

    train[f"{col}ChurnStd"] = train[col].map(churn_std)
    test[f"{col}ChurnStd"] = test[col].map(churn_std)

    freq = pd.concat([train[col], test[col]]).value_counts(normalize=True)

    train[f"{col}Freq"] = train[col].map(freq)
    test[f"{col}Freq"] = test[col].map(freq)

In [10]:
q1 = train["MonthlyCharges"].quantile(0.25)
q3 = train["MonthlyCharges"].quantile(0.75)

for df in [train, test]:

    df["ChargeOutlier"] = (
        (df["MonthlyCharges"] < q1) |
        (df["MonthlyCharges"] > q3)
    ).astype(int)

    df["TenureSquared"] = df["tenure"] ** 2
    df["MonthlyChargesSquared"] = df["MonthlyCharges"] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    df["TenureLog"] = np.log1p(df["tenure"])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]
    df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["SeniorCitizenMonthlyInteraction"] = (
        df["SeniorCitizen"].astype(int) * df["MonthlyCharges"]
    )

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["TenureBucket"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=False
    )

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

    df["Contract_Internet"] = (
        df["Contract"].astype(str) + "_" +
        df["InternetService"].astype(str)
    )

    df["Contract_Payment"] = (
        df["Contract"].astype(str) + "_" +
        df["PaymentMethod"].astype(str)
    )

new_nums = [col for col in df.columns.tolist() if col not in (nums + cats + [target])]

In [11]:
for col in cats + nums:
    tmp = original.groupby(col)[target].mean()
    name = f"OrigP{col}"
    train = train.merge(tmp.rename(name), on=col, how="left")
    test = test.merge(tmp.rename(name), on=col, how="left")
    for df in [train, test]:
        df[name] = df[name].fillna(0.5).astype('float32')
    new_nums.append(name)

In [12]:
X, y = train.drop(target, axis=1), train[target]

In [13]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod', 'Contract_Internet', 'Contract_Payment']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [14]:
import json

with open('xgboost-params.json', 'r') as f:
    xgboost_params = json.load(f)

with open('lightgbm-params.json', 'r') as f:
    lightgbm_params = json.load(f)

In [15]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

oof_xgb = np.zeros(len(X_train_encoded))
test_xgb = np.zeros(len(X_test_encoded))

oof_lgb = np.zeros(len(X_train_encoded))
test_lgb = np.zeros(len(X_test_encoded))

oof_cat = np.zeros(len(X_train_encoded))
test_cat = np.zeros(len(X_test_encoded))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, y_train = X_train_encoded.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X_train_encoded.iloc[valid_idx], y.iloc[valid_idx]

    xgboost_model = XGBClassifier(
        n_estimators=5000,
        **xgboost_params,
        eval_metric='auc',
        random_state=42,
        early_stopping_rounds=300
    )

    xgboost_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=0
    )

    oof_xgb[valid_idx] = xgboost_model.predict_proba(X_valid)[:, 1]
    test_xgb += xgboost_model.predict_proba(X_test_encoded, iteration_range=(0, xgboost_model.best_iteration))[:, 1] / skf.n_splits
    print('XGBoost finished')

    lightgbm_model = LGBMClassifier(
        n_estimators=5000,
        **lightgbm_params,
        objective="binary",
        metric="auc",
        verbosity=-1,
        random_state=42
    )

    lightgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.log_evaluation(100),
            lgb.early_stopping(300)
        ]
    )
    oof_lgb[valid_idx] = lightgbm_model.predict_proba(X_valid)[:, 1]
    test_lgb += lightgbm_model.predict_proba(X_test_encoded, num_iteration=lightgbm_model.best_iteration_)[:, 1] / skf.n_splits
    print('LightGBM finished')

    catboost_model = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.03,
        depth=6,
        eval_metric="AUC",
        verbose=0,
        random_state=42
    )

    catboost_model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=300,
        verbose=False
    )
    
    oof_cat[valid_idx] = catboost_model.predict_proba(X_valid)[:, 1]
    test_cat += catboost_model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits
    print('CatBoost finished')

    print("XGB AUC:", roc_auc_score(y_valid, oof_xgb[valid_idx]))
    print("LGB AUC:", roc_auc_score(y_valid, oof_lgb[valid_idx]))
    print("CAT AUC:", roc_auc_score(y_valid, oof_cat[valid_idx]))


Fold 1
XGBoost finished
Training until validation scores don't improve for 300 rounds
[100]	valid_0's auc: 0.914672
[200]	valid_0's auc: 0.915769
[300]	valid_0's auc: 0.916222
[400]	valid_0's auc: 0.916506
[500]	valid_0's auc: 0.916677
[600]	valid_0's auc: 0.916786
[700]	valid_0's auc: 0.916863
[800]	valid_0's auc: 0.916903
[900]	valid_0's auc: 0.916952
[1000]	valid_0's auc: 0.916955
[1100]	valid_0's auc: 0.916963
[1200]	valid_0's auc: 0.917001
[1300]	valid_0's auc: 0.916991
[1400]	valid_0's auc: 0.916989
[1500]	valid_0's auc: 0.917002
[1600]	valid_0's auc: 0.917006
[1700]	valid_0's auc: 0.916994
[1800]	valid_0's auc: 0.91697
Early stopping, best iteration is:
[1519]	valid_0's auc: 0.917014
LightGBM finished
CatBoost finished
XGB AUC: 0.9171331115624746
LGB AUC: 0.9170144491399639
CAT AUC: 0.916605292119349
Fold 2
XGBoost finished
Training until validation scores don't improve for 300 rounds
[100]	valid_0's auc: 0.915573
[200]	valid_0's auc: 0.916512
[300]	valid_0's auc: 0.917006
[400]

In [16]:
print("XGB CV:", roc_auc_score(y, oof_xgb))
print("LGB CV:", roc_auc_score(y, oof_lgb))
print("CAT CV:", roc_auc_score(y, oof_cat))

XGB CV: 0.9173327234561794
LGB CV: 0.9172199770244975
CAT CV: 0.9168728615142747


In [31]:
train_meta = pd.DataFrame({
    'XGBoost': oof_xgb,
    'LightGBM': oof_lgb,
    'CatBoost': oof_cat
})

test_meta = pd.DataFrame({
    'XGBoost': test_xgb,
    'LightGBM': test_lgb,
    'CatBoost': test_cat
})

In [32]:
train_meta = pd.concat(
    [train_meta, X_train_encoded[['remainder__tenure', 'remainder__MonthlyCharges']]],
    axis=1
)

test_meta = pd.concat(
    [test_meta, X_test_encoded[['remainder__tenure', 'remainder__MonthlyCharges']]],
    axis=1
)

meta_model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=3
)

meta_model.fit(train_meta, y)
final_pred = meta_model.predict_proba(test_meta)[:,1]

In [33]:
submission = pd.read_csv('sample_submission.csv')
submission['Churn'] = final_pred
submission.to_csv('submission.csv', index=False)

In [21]:
blend_preds = 0.4 * test_xgb + 0.4 + test_lgb + 0.2 * test_cat
submission['Churn'] = blend_preds
submission.to_csv('submission.csv', index=False)